## Step 6b — Multi-Horizon Feature Engineering (15, 30, 60 min)

**What changed vs. notebook 02:** the target and lag construction is wrapped in one
function, `build_dataset(base, h)`, so the *same* verified logic produces the 15-min
(h=1), 30-min (h=2) and 60-min (h=4) datasets. Notebook 02 stays as your h=1 record.

**New in this notebook**
1. Target and lags are looked up **by timestamp** (`t + h*15min`, `t - k*15min`). If that exact
   timestamp does not exist the value is NaN and the row is dropped. This is the same rule as your
   `.diff()` gap flags, but it works for any horizon without separate gap logic.
2. Cyclical time-of-day features (`hour_sin`, `hour_cos`) so we can test whether lag features help
   *beyond* simply knowing the time of day.
3. `TARGET_TIME` column, used later for daytime/night evaluation masks and leakage checks.
4. Verification of **all** lags and the target for every horizon (notebook 02 checked lag1 only).

Not leakage: `hour_sin/cos` come from the timestamp at time t, which is known at forecast time.

In [2]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

import pandas as pd
import numpy as np

folder = "/content/drive/MyDrive/Project2_SolarPV/"

Mounted at /content/drive


### Setup — recreate the merged dataset (identical to notebook 02, Steps 1–4)
**Fix: plant-level AC power.** Notebook 01 used `groupby("DATE_TIME").sum()`. That is only correct when all
22 inverters report at a timestamp. In Plant 1, about 3% of timestamps have fewer than 22 inverters (some
in daytime), which makes the plant total look artificially low. Here the plant total is
**mean inverter AC power x 22**, which stays on the same scale when an inverter is missing.

LOAD + MERGE

In [3]:
gen = pd.read_csv(folder + "Plant_1_Generation_Data.csv")
wth = pd.read_csv(folder + "Plant_1_Weather_Sensor_Data.csv")

gen["DATE_TIME"] = pd.to_datetime(gen["DATE_TIME"], format="%d-%m-%Y %H:%M")
wth["DATE_TIME"] = pd.to_datetime(wth["DATE_TIME"], format="%Y-%m-%d %H:%M:%S")

N_INVERTERS = 22

gen_plant = gen.groupby("DATE_TIME").agg(
    DC_POWER=("DC_POWER", "sum"),
    AC_MEAN=("AC_POWER", "mean"),
    DAILY_YIELD=("DAILY_YIELD", "sum"),
    TOTAL_YIELD=("TOTAL_YIELD", "sum"),
    N_INV=("SOURCE_KEY", "nunique"),
).reset_index()

# Plant-level AC power = mean inverter output x 22 (robust to missing inverters)
gen_plant["AC_POWER"] = gen_plant["AC_MEAN"] * N_INVERTERS

short = gen_plant["N_INV"] < N_INVERTERS
daytime = (gen_plant["DATE_TIME"].dt.hour >= 6) & (gen_plant["DATE_TIME"].dt.hour < 18)
print(f"Timestamps with fewer than {N_INVERTERS} inverters: {int(short.sum())} of {len(gen_plant)} "
      f"({int((short & daytime).sum())} of them in daytime)")
gen_plant = gen_plant.drop(columns=["AC_MEAN", "N_INV"])

wth_clean = wth[["DATE_TIME", "AMBIENT_TEMPERATURE", "MODULE_TEMPERATURE", "IRRADIATION"]].copy()
merged = pd.merge(gen_plant, wth_clean, on="DATE_TIME", how="inner")

print("Merged shape:", merged.shape)          # expect (3157, 8) on the real data
print("Missing values:", int(merged.isna().sum().sum()))

Timestamps with fewer than 22 inverters: 101 of 3158 (10 of them in daytime)
Merged shape: (3157, 8)
Missing values: 0


### Build the horizon-aware datasets

- `h` = number of 15-minute steps ahead (1 → 15 min, 2 → 30 min, 4 → 60 min).
- Leakage-prone / redundant columns (`DAILY_YIELD`, `TOTAL_YIELD`, `DC_POWER`) are dropped, as before.
- Rows are dropped (never interpolated) when the target or any lag timestamp does not exist.

In [5]:
STEP = pd.Timedelta(minutes=15)
HORIZONS = {1: "15min", 2: "30min", 4: "60min"}   # steps ahead -> label
N_LAGS = 3

base = (merged.drop(columns=["DAILY_YIELD", "TOTAL_YIELD", "DC_POWER"])
              .sort_values("DATE_TIME").reset_index(drop=True))
assert base["DATE_TIME"].is_unique, "Duplicate timestamps after aggregation!"

ac_by_time = base.set_index("DATE_TIME")["AC_POWER"]


def build_dataset(base, h, n_lags=N_LAGS):
    """Return the cleaned dataset for forecast horizon h (in 15-min steps)."""
    df = base.copy()
    t = df["DATE_TIME"]

    # Lag features: AC_POWER at exactly t-1, t-2, t-3 steps (NaN if that timestamp is missing)
    for k in range(1, n_lags + 1):
        df[f"AC_POWER_lag{k}"] = ac_by_time.reindex(t - k * STEP).to_numpy()

    # Target: AC_POWER at exactly t + h steps (NaN if that timestamp is missing)
    df["TARGET_TIME"] = t + h * STEP
    df["TARGET"] = ac_by_time.reindex(df["TARGET_TIME"]).to_numpy()

    # Time-of-day features (known at time t, so no leakage)
    minutes = t.dt.hour * 60 + t.dt.minute
    df["hour_sin"] = np.sin(2 * np.pi * minutes / 1440)
    df["hour_cos"] = np.cos(2 * np.pi * minutes / 1440)

    needed = ["TARGET"] + [f"AC_POWER_lag{k}" for k in range(1, n_lags + 1)]
    n_before = len(df)
    df = df.dropna(subset=needed).reset_index(drop=True)
    return df, n_before - len(df)


datasets = {}
for h, label in HORIZONS.items():
    d, n_dropped = build_dataset(base, h)
    datasets[h] = d
    print(f"h={h} ({label}): {len(d)} rows kept, {n_dropped} dropped, columns={d.shape[1]}")

print("\nColumns:", datasets[1].columns.tolist())

h=1 (15min): 3113 rows kept, 44 dropped, columns=12
h=2 (30min): 3105 rows kept, 52 dropped, columns=12
h=4 (60min): 3092 rows kept, 65 dropped, columns=12

Columns: ['DATE_TIME', 'AC_POWER', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION', 'AC_POWER_lag1', 'AC_POWER_lag2', 'AC_POWER_lag3', 'TARGET_TIME', 'TARGET', 'hour_sin', 'hour_cos']


### Sanity check against your notebook-02 result (h=1)

**Expected:** the new method keeps **every row** your notebook 02 kept, with identical values,
plus **about one extra row per timestamp gap**.

**Why:** in notebook 02, `lags_valid` includes `gap_before.shift(3)`. A gap flag at row *i* means the step
(i-1 → i) is broken. The lag window of row *t* needs steps (t-3→t-2), (t-2→t-1), (t-1→t), i.e. flags at
t, t-1, t-2 only. The `shift(3)` term (step t-4→t-3) is not part of the window, so notebook 02 drops one
perfectly valid row after every gap. It is a conservative error: no leakage, no wrong values, just ~10 rows
lost. (54 rows dropped = 10 gaps x 5 rows + 3 start rows + 1 end row; the correct count is 10 x 4 + 3 + 1 = 44.)

In [6]:
orig = base.copy()
orig["time_diff"] = orig["DATE_TIME"].diff()
orig["gap_before"] = orig["time_diff"] != STEP
orig.loc[0, "gap_before"] = False
n_gap_events = int(orig["gap_before"].sum())
orig["TARGET_old"] = orig["AC_POWER"].shift(-1)
orig["target_gap_valid"] = ~orig["gap_before"].shift(-1, fill_value=True)
for k in (1, 2, 3):
    orig[f"AC_POWER_lag{k}"] = orig["AC_POWER"].shift(k)
orig["lags_valid"] = (
    (~orig["gap_before"]) & (~orig["gap_before"].shift(1, fill_value=True))
    & (~orig["gap_before"].shift(2, fill_value=True)) & (~orig["gap_before"].shift(3, fill_value=True))
)
orig = orig[orig["target_gap_valid"] & orig["lags_valid"] & orig["TARGET_old"].notna()]

new_h1 = datasets[1]
cmp = orig.merge(new_h1, on="DATE_TIME", how="left", suffixes=("_old", "_new"), indicator=True)
assert (cmp["_merge"] == "both").all(), "A row kept by notebook 02 is missing in the new dataset!"
assert np.allclose(cmp["TARGET_old"], cmp["TARGET"])
for k in (1, 2, 3):
    assert np.allclose(cmp[f"AC_POWER_lag{k}_old"], cmp[f"AC_POWER_lag{k}_new"])

print(f"Gap events in timestamps:        {n_gap_events}")
print(f"Rows kept by notebook 02 logic:  {len(orig)}")
print(f"Rows kept by new logic (h=1):    {len(new_h1)}")
print(f"Extra rows recovered:            {len(new_h1) - len(orig)}  (expect about {n_gap_events})")
print("All rows from notebook 02 are present with identical values. OK")

Gap events in timestamps:        10
Rows kept by notebook 02 logic:  3103
Rows kept by new logic (h=1):    3113
Extra rows recovered:            10  (expect about 10)
All rows from notebook 02 are present with identical values. OK


### Independent verification for every horizon (target + all lags)

Same idea as your Part 7 check, but done with a plain Python dictionary lookup (a different
code path from `reindex`) and extended to lag2, lag3 and all horizons.

In [7]:
lookup = dict(zip(base["DATE_TIME"], base["AC_POWER"]))

for h, d in datasets.items():
    exp_target = d["DATE_TIME"].map(lambda t: lookup.get(t + h * STEP, np.nan))
    assert (exp_target.to_numpy() == d["TARGET"].to_numpy()).all(), f"target mismatch at h={h}"
    for k in (1, 2, 3):
        exp_lag = d["DATE_TIME"].map(lambda t: lookup.get(t - k * STEP, np.nan))
        assert (exp_lag.to_numpy() == d[f"AC_POWER_lag{k}"].to_numpy()).all(), f"lag{k} mismatch at h={h}"
    assert (d["TARGET_TIME"] - d["DATE_TIME"] == h * STEP).all()
    print(f"h={h}: target and lag1-3 verified, 0 mismatches ({len(d)} rows)")

h=1: target and lag1-3 verified, 0 mismatches (3113 rows)
h=2: target and lag1-3 verified, 0 mismatches (3105 rows)
h=4: target and lag1-3 verified, 0 mismatches (3092 rows)


In [8]:
for h, d in datasets.items():
    d.to_csv(folder + f"df_clean_h{h}.csv", index=False)
print("Saved df_clean_h1.csv, df_clean_h2.csv, df_clean_h4.csv")

Saved df_clean_h1.csv, df_clean_h2.csv, df_clean_h4.csv
